# Step 2 — Train Random Forest crop classifier with fixed hyperparameters

**Input:** merged table from `data/prepared/`  
**What it does:**
- filters to crop-only Level-2 classes
- selects VV/VH time-window features
- trains a Random Forest with fixed hyperparameters
- evaluates on a stratified validation split
- saves the trained model + feature list into `models/`

In [ ]:
import json
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_PREP = REPO_ROOT / "data" / "prepared"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Filenames (edit ONLY these if your filenames differ)
data_file = DATA_PREP / "S1_point_all_10d_10m_20180101-20180731_Stratum1_VV-VH.csv"
classes_file = DATA_PREP / "LUCAS_2018_Copernicus_attributes_cropmap_level1-2_FROM_EXPORTS.csv"

# Friendly errors if files are missing
assert data_file.exists(), f"Missing dataset file: {data_file}"
assert classes_file.exists(), f"Missing classes file: {classes_file}"

print("Using data_file:", data_file)
print("Using classes_file:", classes_file)
print("Saving outputs to:", MODELS_DIR)

In [ ]:
class_table = pd.read_csv(classes_file)
print(class_table)


In [ ]:
# Load class legend table 

classes_L1 = class_table["level_1"].dropna().unique().tolist()
classes_L2 = class_table["level_2"].dropna().unique().tolist()

df = pd.read_csv(data_file, dtype={'level_1': int, 'level_2': int})
print(f"Loaded dataset with shape {df.shape}")


In [ ]:
#official Level-2 set:
L2_official = {211,212,213,214,215,216,217,218,219,221,222,223,230,231,232,233,240,250,290}



# sremove non crop labled data
bad_L1 = {100,300,500,600}
classes_L2 = [c for c in classes_L2 if c in L2_official]

print(f"Classes in level_1: {classes_L1}")
print(f"Classes in level_2: {classes_L2}")
print(df.head())

In [ ]:
df['Classif'] = df['level_2']  # working label for classification (detailed crop types)
if classes_L2:
    df = df[df['Classif'].isin(classes_L2)]
print(f"Data after filtering to crop classes: {df.shape}")



In [ ]:

feature_regex = r'(((?<![\w\d])VH_)|((?<![\w\d])VV_))'  
feature_regex += r'(2018(0[1-7]))'  

X = df.filter(regex=feature_regex)
y = df['Classif']

print(f"Selected features matrix X shape: {X.shape}")
print(f"Selected target vector y shape: {y.shape}")
# Check a quick summary of class distribution
print("Class distribution in y:")
print(y.value_counts())
# feature names used in training
feature_names = list(X.columns)
feat_path = MODELS_DIR / "RF_fixed_params_feature_names.json"
with open(feat_path, "w") as f:
    json.dump(feature_names, f, indent=2)
print(f"Saved training feature list to {feat_path}")


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

rfc = RandomForestClassifier(n_jobs=-1)
print("Training Random Forest with fixed hyperparameters on the training split...")
rfc.fit(X_train, y_train)
print("Training complete.")
print("Hyperparameters used:")
print(rfc.get_params())


In [ ]:
y_pred = rfc.predict(X_val)
print("Accuracy on validation set: {:.2f}%".format(100 * (y_pred == y_val).mean()))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))


In [ ]:
rf_model = final_model
importances = rf_model.feature_importances_
feature_names = X.columns
top_indices = np.argsort(importances)[::-1][:10]  # indices of top 10 important features
print("Top 10 important features:")
for idx in top_indices:
    print(f"{feature_names[idx]}: {importances[idx]:.4f}")
